# Chapter 8: RAG Generation — The Answer Layer

## Topic 2: Citation and Source Attribution

---

### 1. Concept, Intuition, and Why It Exists

- A RAG answer's credibility depends on the customer (or a compliance reviewer) being able to verify where a claim came from. "Premature withdrawal incurs a 1% penalty" is a materially different statement when it's traceable to `04_FD_Policy_Reference.pdf` versus when it's an unattributed claim the model could have hallucinated.
- Citation exists to close the gap between "the model said something plausible" and "the model said something we can verify against a specific source document" — this is the mechanism that makes Topic 4's hallucination detection tractable: if every claim is supposed to trace back to a cited source, you can programmatically check whether it actually does.
- For this project specifically, an NBFC handling regulated financial information, citation is not a nice-to-have UX feature — it's close to a compliance requirement. A customer disputing a penalty calculation needs the underlying policy document identified, not just an LLM's paraphrase.

---

### 2. Internal Working — Step by Step

1. **Source tagging at construction time** (Topic 1's output): each chunk in the context block carries an explicit, parseable marker — `[Source: 01_FD_FAQ.pdf]` — that the model can reference in its answer.
2. **Prompting the model to cite**: the system prompt explicitly instructs the model to attribute every factual claim to its source marker, e.g. "When you state a fact drawn from the context, cite it using the exact source marker shown above the relevant passage."
3. **Two citation strategies, differing in when attribution happens**:
   - **Inline citation**: the model outputs citations as part of its natural-language answer, e.g. "The penalty is 1% [Source: 04_FD_Policy_Reference.pdf]." — simplest to implement, but citation accuracy depends entirely on the model correctly associating each claim with its source during generation.
   - **Structured citation via tool use**: the model's response is a structured object (via Claude's tool-use mechanism, already established in this project's `Chapter_3/1_Agentic_AI.ipynb` agent pattern) with a separate `sources` field listing which source markers were used — more reliable to parse programmatically, and reuses this project's existing tool-calling infrastructure rather than introducing a new mechanism.
4. **Post-hoc verification (the check that makes citation trustworthy, not just decorative)**: after generation, verify that every cited source marker actually appears in the context block that was sent — a citation to a source that was never in context is a definitive signal of model error, independent of whether the underlying factual claim happens to be true.
5. **Surfacing citations to the end consumer**: whether the citation is shown directly to a customer, logged for a human agent's review, or used purely internally for the hallucination check in Topic 4, depends on the deployment context — this project's cascade (Chapter 1) suggests customer-facing surfacing is optional, but internal logging for every generated answer is close to mandatory for a regulated domain.

---

### 3. How It Is Implemented in This Project

- Building directly on this project's existing tool-use pattern (`TOOLS`, `input_schema`, `client.messages.create(tools=TOOLS, ...)` from Chapter 3): a new terminal tool, `finalize_answer_with_citations`, mirrors the design of the existing `finalize_classification` tool — a structured way for the model to declare it's done, but now carrying both an answer and a list of cited sources as required fields in its `input_schema`.

```python
ANSWER_TOOLS = [
    {
        "name": "finalize_answer_with_citations",
        "description": "Call this when you have a complete answer to the customer's question, grounded in the provided context.",
        "input_schema": {
            "type": "object",
            "properties": {
                "answer": {"type": "string", "description": "The final answer to the customer."},
                "sources_used": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Every source marker (e.g. '01_FD_FAQ.pdf') that was actually used to construct the answer."
                },
            },
            "required": ["answer", "sources_used"],
        },
    }
]

RAG_SYSTEM_PROMPT = """You answer customer questions about Fixed Deposits using ONLY the
provided context. Every factual claim in your answer must be traceable to a source shown
in the context. When you are ready to respond, call finalize_answer_with_citations with
your answer and the exact list of source markers you actually relied on."""


def verify_citations(sources_used: list, context_sources: set) -> dict:
    """Post-hoc check: every cited source must have actually been in the context sent."""
    invalid_citations = [s for s in sources_used if s not in context_sources]
    return {
        "valid": len(invalid_citations) == 0,
        "invalid_citations": invalid_citations,
    }
```

- The `context_sources` set passed to `verify_citations()` comes directly from Topic 1's `build_context_block()` — every chunk included there has a known source, making this check a simple set-membership operation, not a semantic verification (that's Topic 4's job).

---

### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Citation to a source that exists but wasn't in this specific context**: a model might cite `01_FD_FAQ.pdf` generically because it's a plausible-sounding source name, without that specific chunk actually having been included in this request's context block — `verify_citations()` catches exactly this by checking against the actual context sent, not against the full knowledge base.
- **Partial or vague citation**: a model citing "the FAQ" instead of the exact source marker string breaks programmatic verification — the system prompt must be explicit about the exact required format, and citation parsing should be tolerant of near-matches while still flagging them for review, not silently accepting or silently failing.
- **Over-citation and under-citation as opposite failure modes**: citing every sentence, even trivial ones, adds noise without adding trust; citing nothing while making several factual claims defeats the purpose entirely — the system prompt should specify what counts as a "factual claim" requiring citation.
- **Monitoring**: track the citation-verification failure rate (from `verify_citations()`) as a first-class production metric, separate from hallucination detection — a rising invalid-citation rate is often an earlier, cheaper-to-detect warning sign than downstream hallucination checks, since it's a simple parsing check, not a semantic one.
- **Cost and latency**: structured citation via tool use has effectively the same cost profile as this project's existing tool-use agent pattern — no meaningful additional latency or token cost beyond what the agent loop already incurs, since it reuses existing infrastructure rather than adding a second model call.
- **Security**: citation markers are strings extracted from the model's own output — never trust a citation string to reference a file path or perform a lookup without validation; treat it purely as a label to check against the known `context_sources` set, never as an argument to a file-system operation or database query directly.
- **Deployment**: `verify_citations()` should run synchronously immediately after the model's response, before the answer is shown to any consumer (customer or human agent) — this is a cheap, fast check that belongs in the critical path, unlike Topic 4's more expensive hallucination detection, which may run asynchronously or on a sample.

---

### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Inline citation vs. structured tool-based citation**: inline citation is simpler to prompt for and more natural to read in a customer-facing surface, but harder to reliably parse and verify programmatically — a citation embedded mid-sentence requires text parsing (regex or similar) with all the brittleness that implies. Structured citation via tool use, already this project's established pattern, is more reliable to verify but requires designing the answer to be reconstructed from structured fields if a customer-facing natural-language surface is still needed.
- **Sentence-level vs. answer-level citation granularity**: citing which sources contributed to the answer as a whole (this project's `sources_used` list design) is simpler to implement and verify than citing which source supports each individual sentence — the latter gives finer-grained trust signals (useful for Topic 4's grounding checks) at the cost of a more complex prompt and structured-output schema. For this project's initial implementation, answer-level citation is the pragmatic starting point; sentence-level citation is a natural refinement once answer-level citation is validated and the added complexity is justified by measured need.
- **Should verification failures block the answer, or just flag it?**: for a regulated financial domain, a hard block (refuse to surface an answer with unverifiable citations, route to human review instead) is more defensible than silently surfacing a flagged-but-unblocked answer — this mirrors the existing `refuse_out_of_scope` pattern already established in Chapter 3's agent design philosophy.

---

### 6. Alternatives and When to Use Each

- **No citation at all**: acceptable only for the lowest-stakes, most exploratory use cases — not appropriate for this project's regulated financial domain given the compliance and dispute-resolution considerations already discussed.
- **Inline natural-language citation**: reasonable for customer-facing surfaces where a clean, readable answer matters more than perfect programmatic verifiability, provided a best-effort parsing/verification layer still exists.
- **Structured tool-based citation (this project's chosen approach)**: the right default given the project's existing tool-use infrastructure and the need for reliable, programmatic verification for both compliance logging and Topic 4's hallucination detection.
- **Sentence-level attribution (e.g. via a separate model pass mapping each sentence to a source)**: reserved for scenarios where answer-level citation has been validated as insufficient — e.g. long, multi-fact answers where knowing *which part* is grounded and which isn't materially changes how a human reviewer or downstream system should act.

---

### 7. Common Mistakes and Production Failures

- Trusting citations without post-hoc verification against the actual context sent — a model can cite a plausible-sounding but incorrect source, and without verification this goes undetected.
- Using citation strings from model output directly in file-system or database operations without validation, creating an injection risk.
- Conflating "the citation is valid" (the source was in context) with "the answer is grounded" (the claim actually matches what the source says) — citation verification is necessary but not sufficient; Topic 4 covers the deeper, semantic grounding check.
- Designing a citation format the model can't reliably reproduce (e.g. requiring exact page numbers when chunks don't carry that metadata) — the citation schema must only ask for what the pipeline actually attaches to each chunk.
- Not logging citation-verification failures separately from other error types, losing an early, cheap warning signal that could have caught problems before they reached more expensive downstream checks.

---

### 8. Lead-Level Interview Questions

**Basic:**

**Q: Why is citation important for a RAG system, beyond general "trustworthiness"?**
A: Citation makes it possible to programmatically verify whether a model's claim is actually grounded in a specific, checkable source, rather than relying on the claim's surface plausibility. For a regulated financial domain, this is close to a compliance requirement — customers and reviewers need to trace a stated fact (like a penalty rate) back to the exact policy document, not just trust the model's paraphrase.

**Q: What is the difference between citation verification and hallucination detection?**
A: Citation verification is a cheap, syntactic check — does the cited source marker actually appear in the context that was sent to the model? Hallucination detection (Topic 4) is a more expensive, semantic check — does the model's claim actually match what the cited source says? A citation can be valid (the source was genuinely in context) while the claim attributed to it is still wrong or unsupported — citation verification alone does not catch this.

**Intermediate:**

**Q: Why does this project use structured tool-based citation rather than inline natural-language citation?**
A: The project already has an established tool-use agent pattern (Chapter 3's `finalize_classification` and `refuse_out_of_scope` tools) — extending this pattern with a `finalize_answer_with_citations` tool reuses existing infrastructure and produces a structured `sources_used` field that's trivial to programmatically verify via simple set membership, versus inline citation which requires parsing free text and is more brittle to variations in how the model phrases its output.

**Q: How would you design the verification check to distinguish "the model cited a source that was never in context" from "the model's claim doesn't match what the cited source actually says"?**
A: These require two separate checks at two separate cost tiers. The first — citation validity — is a cheap set-membership check: is the cited source marker present in the known `context_sources` set for this specific request? This should run synchronously on every request. The second — semantic grounding — requires either an NLI-style entailment model, an LLM-as-judge call, or embedding-similarity comparison between the claim and the cited passage, all of which are more expensive; this is Topic 4's concern and may run on a sample or asynchronously rather than blocking every response.

**Advanced:**

**Q: Design the failure-handling policy for when `verify_citations()` detects an invalid citation in a customer-facing RAG response for this project.**
A: Given the regulated financial domain, an invalid citation should not silently pass through to the customer. The response should be blocked from customer-facing surfacing and routed to a human-review queue, mirroring the existing `refuse_out_of_scope` pattern's philosophy of making failures explicit and structured rather than silent. The specific invalid citation(s) and the full context that was sent should be logged for review, both to fix the immediate case and to identify whether a systematic prompt or context-construction issue is causing repeated failures. A rising rate of invalid citations for a specific query pattern is also a signal worth feeding back into Chapter 7's evaluation harness — it may indicate the underlying retrieval or context construction has a gap for that pattern.

**Q: A teammate argues that sentence-level citation is strictly better than answer-level citation and should be the default. How do you respond?**
A: Sentence-level citation provides finer-grained trust signals — you can tell exactly which claims are grounded and which aren't, rather than a single verified/unverified flag for the whole answer. But it requires a more complex prompt and output schema, is harder for the model to reliably produce correctly, and adds latency and potential failure surface. For most of this project's query types — short, single-fact-focused customer emails — answer-level citation likely captures most of the practical value at much lower complexity. Sentence-level citation becomes worth the added cost specifically for longer, multi-fact answers where partial grounding failure (some claims correct, others not) is a real, distinguishable risk — this should be validated with actual production data before being adopted as a default, not assumed superior in all cases.

**Scenario-based:**

**Q: In production, `verify_citations()` starts flagging a rising rate of invalid citations specifically for questions about a newly-launched FD product. Diagnose.**
A: This pattern — a spike isolated to a specific new topic — strongly suggests the model is citing a plausible-sounding but non-existent or incorrect source for content it either doesn't have good grounding for, or where the actual correct source wasn't retrieved and included in context at all (a Chapter 7 retrieval gap, not a Chapter 8 generation gap). First check whether documentation for the new product was actually ingested and is being retrieved (cross-reference against Chapter 7 Topic 9's evaluation harness, extended with queries about the new product). If retrieval is genuinely finding the right chunks but the model still cites incorrectly, the issue is more likely a prompting or model-behavior problem specific to unfamiliar content — worth testing whether being more explicit in the system prompt about strict citation-to-provided-context-only behavior reduces the rate.

---

### 9. Hidden Concepts and Prerequisites

- **Citation verification is a special case of output validation** (a general pattern that recurs across this project — structured tool schemas already provide a first layer of output validation by construction, and citation verification is a second, semantic-adjacent layer on top of that structural validation).
- **The relationship between citation and RAGAS's Context Precision metric** (forward reference to Chapter 14): Context Precision measures whether the retrieved context that was actually used is relevant — citation data (which sources the model says it used) is a direct input to computing this metric at scale, connecting this topic's mechanism to the more formal end-to-end RAG evaluation covered later.
- **Citation as an interpretability tool, not just a trust mechanism**: beyond customer-facing trust, citation data aggregated across many requests reveals which knowledge base documents are actually being relied upon in practice versus which are rarely cited despite being retrieved — useful signal for knowledge base maintenance and Chapter 4's ingestion prioritization.

---

### 10. Revision Summary

> Citation attributes each factual claim in a RAG answer to a specific, verifiable source from the retrieved context, closing the gap between "plausible-sounding" and "actually grounded." This project implements structured citation via a `finalize_answer_with_citations` tool, extending the existing Chapter 3 agent tool-use pattern rather than relying on inline natural-language citation, because it produces a reliably parseable `sources_used` field. Citation verification (checking cited sources actually appeared in the context sent) is a cheap, necessary-but-not-sufficient syntactic check, distinct from the more expensive semantic grounding check covered in Topic 4 — a citation can be valid while the underlying claim is still wrong. Invalid citations in a regulated financial domain should block customer-facing surfacing and route to human review, not silently pass through, and should be logged and monitored as an early warning signal distinct from downstream hallucination metrics.

---